# F04: Real-Time Streaming

Stream synthetic capital markets trade events through Spindle's streaming engine with burst windows, out-of-order delivery, and anomaly injection.

## What You'll Learn
- Generate capital markets data (trades, quotes, positions)
- Use `SpindleStreamer` with `StreamConfig` to control streaming behavior
- Send events through `ConsoleSink` and `FileSink`
- Configure burst windows for traffic spikes
- Inject out-of-order delivery and anomalies
- Use `FabricStreamWriter` for Eventstream integration (overview)

## Prerequisites
- Python 3.10+
- `sqllocks-spindle` installed
- For Eventstream: `sqllocks-spindle[streaming]` extra

## Time Estimate
~20 minutes

In [ ]:
# %pip install sqllocks-spindle[streaming]

## Step 1: Stream Trade Events to Console

**What we're about to do:** Use `SpindleStreamer` to generate capital markets data and stream trade events to a `ConsoleSink`. This lets you see the events as they would flow through a real pipeline.

**Why this matters:** Real-time intelligence (RTI) in Fabric requires a stream of events. Spindle generates events with proper timestamps, sequence numbers, and metadata -- exactly what Eventstream expects.

**Key concepts:**
- `SpindleStreamer` generates data on-demand (or accepts pre-generated tables) and converts rows to event dicts
- Each event gets `_spindle_table`, `_spindle_seq`, and `_spindle_event_time` metadata fields
- `StreamConfig(max_events=N)` caps the number of events emitted

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.streaming import (
    SpindleStreamer, StreamConfig, ConsoleSink
)

# First, generate capital markets data
spindle = Spindle()
from sqllocks_spindle.domains.financial import FinancialDomain
result = spindle.generate(domain=FinancialDomain(), scale="small", seed=42)

print(f"Generated tables: {result.table_names}")
print(f"Total rows: {sum(result.row_counts.values()):,}")

# Stream the first available table to console (limited to 10 events)
streamer = SpindleStreamer(
    tables=result.tables,
    sink=ConsoleSink(),
    config=StreamConfig(max_events=10),
    seed=42,
)

first_table = result.table_names[0]
stream_result = streamer.stream(first_table)
print(f"\n{stream_result}")

## Step 2: Stream to a File with FileSink

**What we're about to do:** Use `FileSink` to write streaming events to a JSONL file. This is how you'd capture events locally for testing before connecting to Eventstream.

**What to expect:** A `.jsonl` file where each line is a JSON event object with all table columns plus Spindle metadata fields.

In [ ]:
from sqllocks_spindle.streaming import FileSink
import json

# Stream 200 events to a JSONL file
file_sink = FileSink("streaming_events.jsonl", mode="w")
streamer = SpindleStreamer(
    tables=result.tables,
    sink=file_sink,
    config=StreamConfig(max_events=200),
    seed=42,
)

stream_result = streamer.stream(first_table)
print(f"Streamed {stream_result.events_sent} events to streaming_events.jsonl")
print(f"Elapsed: {stream_result.elapsed_seconds:.3f}s")
print(f"Throughput: {stream_result.events_per_second_actual:,.0f} events/sec")

# Show the first event
with open("streaming_events.jsonl") as f:
    first_event = json.loads(f.readline())
print(f"\nSample event keys: {list(first_event.keys())}")
print(json.dumps(first_event, indent=2, default=str)[:500])

## Step 3: Burst Windows and Out-of-Order Delivery

**What we're about to do:** Configure a `BurstWindow` that creates a 10x traffic spike at t=5s lasting 3s, and enable out-of-order delivery for 10% of events.

**Why this matters:** Real streaming systems experience traffic bursts (flash sales, market opens) and network-induced reordering. Your KQL queries and Eventstream processing need to handle both.

**Key concepts:**
- `BurstWindow(start_offset_seconds=5, duration_seconds=3, multiplier=10.0)` -- 10x rate at t=5-8s
- `out_of_order_fraction=0.10` -- 10% of events arrive late (shifted forward in sequence)
- `TimePattern.business_hours()` -- higher traffic during 8am-6pm weekdays

In [ ]:
from sqllocks_spindle.streaming import BurstWindow, TimePattern

# Configure streaming with burst window and out-of-order delivery
config = StreamConfig(
    max_events=500,
    out_of_order_fraction=0.10,          # 10% of events arrive out of order
    out_of_order_max_delay_slots=20,     # Late events shift up to 20 positions
    burst_windows=[
        BurstWindow(
            start_offset_seconds=5,      # Burst starts at t=5s
            duration_seconds=3,          # Lasts 3 seconds
            multiplier=10.0,             # 10x normal rate during burst
        ),
    ],
    time_pattern=TimePattern.business_hours(),  # Higher traffic during work hours
    realtime=False,  # Fast mode (no rate limiting) for this demo
)

streamer = SpindleStreamer(
    tables=result.tables,
    sink=FileSink("burst_events.jsonl", mode="w"),
    config=config,
    seed=42,
)

stream_result = streamer.stream(first_table)
print(f"Events sent:     {stream_result.events_sent:,}")
print(f"Out-of-order:    {stream_result.out_of_order_count}")
print(f"Elapsed:         {stream_result.elapsed_seconds:.3f}s")

## Step 4: Anomaly Injection

**What we're about to do:** Add an `AnomalyRegistry` with point anomalies (single outlier values), contextual anomalies (normal value in wrong context), and collective anomalies (suspicious patterns across multiple events).

**Why this matters:** If you're building anomaly detection in KQL or Spark, you need known anomalies in your test data to validate detection logic. Spindle labels each anomaly so you can verify your rules catch them.

In [ ]:
from sqllocks_spindle.streaming import (
    AnomalyRegistry, PointAnomaly, ContextualAnomaly, CollectiveAnomaly
)

# Build an anomaly registry with three types of anomalies
registry = AnomalyRegistry()
registry.register(PointAnomaly(fraction=0.05))        # 5% of events get outlier values
registry.register(ContextualAnomaly(fraction=0.03))   # 3% get contextually wrong values
registry.register(CollectiveAnomaly(fraction=0.02))   # 2% form suspicious clusters

# Stream with anomaly injection and labeling
config_anomaly = StreamConfig(
    max_events=500,
    label_anomalies=True,  # Keep _spindle_is_anomaly column in output
)

streamer = SpindleStreamer(
    tables=result.tables,
    sink=FileSink("anomaly_events.jsonl", mode="w"),
    config=config_anomaly,
    anomaly_registry=registry,
    seed=42,
)

stream_result = streamer.stream(first_table)
print(f"Events sent:     {stream_result.events_sent:,}")
print(f"Anomalies:       {stream_result.anomaly_count}")
print(f"Anomaly rate:    {stream_result.anomaly_count / max(stream_result.events_sent, 1) * 100:.1f}%")

## Step 5: FabricStreamWriter for Eventstream Integration

**What we're about to do:** Show the high-level `FabricStreamWriter` API that wraps domain generation + streaming + Eventstream sink into a single call. This is the fastest path from "I need test data" to events flowing in Fabric Eventstream.

**Key concept:** `FabricStreamWriter` requires an Event Hub-compatible connection string from your Eventstream custom endpoint. It handles domain generation, streaming engine setup, and Event Hub protocol internally.

In [ ]:
# FabricStreamWriter is the high-level convenience API for Eventstream.
# Uncomment and fill in your Eventstream connection string to use it.

# from sqllocks_spindle.fabric import FabricStreamWriter
#
# fabric_writer = FabricStreamWriter(
#     connection_string="Endpoint=sb://YOUR_EVENTSTREAM.servicebus.windows.net/;SharedAccessKeyName=...;SharedAccessKey=...",
#     domain="financial",
#     table=first_table,
#     scale="small",
# )
#
# result = fabric_writer.stream(max_events=1000, rate=100.0)
# print(f"Streamed {result.events_sent:,} events in {result.elapsed_seconds:.1f}s")

print("FabricStreamWriter usage (requires Eventstream connection string):")
print()
print("  from sqllocks_spindle.fabric import FabricStreamWriter")
print()
print('  writer = FabricStreamWriter(')
print('      connection_string="Endpoint=sb://...",')
print('      domain="financial",')
print(f'      table="{first_table}",')
print('  )')
print('  result = writer.stream(max_events=1000, rate=100.0)')

## What's Next?

You've streamed synthetic events with burst windows, out-of-order delivery, and anomaly injection.

- **F05: Chaos Pipeline Testing** -- Inject data quality chaos into batch data and test validation gates
- **F01: Medallion Architecture** -- Build a Bronze/Silver/Gold pipeline that could ingest these streaming events
- **F06: Semantic Model Builder** -- Create a Power BI semantic model on top of your streamed and aggregated data